In [0]:
%sql
SELECT * FROM bi_subject.colombia_economics.filtered_monthly LIMIT 5;

MonthlyDate,InflationYOY,TRMAvg,OccupationRate,UnemploymentRate,PolicyRate,ColcapIndex,MonthlyReturn
2018-01,3.679528,2868.57,57.3864,12.1374,4.5,1558.18,2.941895418359586
2018-02,3.372119,2860.25,57.8333,11.1882,4.5,1478.33,-5.124568406730933
2018-03,3.139859,2847.93,58.7333,9.7716,4.5,1455.52,-1.5429572558224414
2018-04,3.127613,2766.29,59.8379,9.7292,4.25,1565.56,7.56018467626689
2018-05,3.156785,2861.83,59.2541,9.9795,4.25,1546.71,-1.2040420041390831


## INFLATION: How did inflation evolve between 2018 and 2026, and when were its major peaks?

In [0]:
%sql
SELECT 
    YEAR(MonthlyDate) AS Year, AVG(InflationYOY) AS AvgInflation, 
    MAX(InflationYOY) AS MaxInflation, MIN(InflationYOY) AS MinInflation
FROM bi_subject.colombia_economics.filtered_monthly
WHERE InflationYOY IS NOT NULL
GROUP BY YEAR(MonthlyDate)
ORDER BY Year;

Year,AvgInflation,MaxInflation,MinInflation
2018,3.2414955000000005,3.679528,3.098241
2019,3.517890833333334,3.85561,3.01408
2020,2.534736666666667,3.85593,1.48947
2021,3.494290833333333,5.62267,1.50655
2022,10.150811666666668,13.12247,6.9446
2023,11.774830833333333,13.34433,9.27605
2024,6.62876,8.34657,5.19736
2025,5.142500833333334,5.51223,4.82007
2026,5.7670825,6.24169,5.29003


## Exchange rate: How did the TRM evolve between 2018 and 2026, particularly around major economic shocks?

In [0]:
%sql
SELECT
    YEAR(TO_DATE(MonthlyDate, 'yyyy-MM')) AS Year,
    ROUND(AVG(TRMAvg), 2) AS AverageTRM,
    ROUND(MIN(TRMAvg), 2) AS MinimumTRM,
    ROUND(MAX(TRMAvg), 2) AS MaximumTRM
FROM bi_subject.colombia_economics.filtered_monthly
WHERE TRMAvg IS NOT NULL
GROUP BY YEAR(TO_DATE(MonthlyDate, 'yyyy-MM'))
ORDER BY Year;

Year,AverageTRM,MinimumTRM,MaximumTRM
2018,2955.81,2766.29,3218.55
2019,3280.01,3115.84,3433.31
2020,3692.76,3311.19,3977.39
2021,3741.8,3491.32,3963.13
2022,4253.03,3792.98,4926.66
2023,4327.55,3948.21,4803.11
2024,4071.18,3867.98,4408.65
2025,4053.02,3786.18,4307.57
2026,3541.07,3130.87,3717.56


## Monetary policy: How did Banco de la República's policy rate respond to changes in inflation?

In [0]:
%sql
SELECT
    CASE
        WHEN InflationYOY < 3 THEN 'Low Inflation'
        WHEN InflationYOY < 5 THEN 'Moderate Inflation'
        ELSE 'High Inflation'
    END AS InflationLevel,
    COUNT(*) AS Months,
    ROUND(AVG(InflationYOY), 2) AS AverageInflation, ROUND(AVG(PolicyRate), 2) AS AveragePolicyRate,
    ROUND(MIN(PolicyRate), 2) AS MinimumPolicyRate, ROUND(MAX(PolicyRate), 2) AS MaximumPolicyRate
FROM bi_subject.colombia_economics.filtered_monthly
WHERE InflationYOY IS NOT NULL
  AND PolicyRate IS NOT NULL
GROUP BY
    CASE
        WHEN InflationYOY < 3 THEN 'Low Inflation'
        WHEN InflationYOY < 5 THEN 'Moderate Inflation'
        ELSE 'High Inflation'
    END
ORDER BY AverageInflation;

InflationLevel,Months,AverageInflation,AveragePolicyRate,MinimumPolicyRate,MaximumPolicyRate
Low Inflation,12,1.86,2.06,1.75,3.25
Moderate Inflation,36,3.61,4.11,1.75,9.25
High Inflation,56,8.07,10.17,2.5,13.25


## Labor market & GDP: How did unemployment, occupation, and economic growth evolve before and after COVID-19?

In [0]:
%sql
WITH labor AS (
    SELECT
        CASE
            WHEN YEAR(TO_DATE(MonthlyDate, 'yyyy-MM')) <= 2019
                THEN 'Pre-COVID'
            WHEN YEAR(TO_DATE(MonthlyDate, 'yyyy-MM')) = 2020
                THEN 'COVID'
            ELSE 'Post-COVID'
        END AS Period,
        OccupationRate,
        UnemploymentRate
    FROM bi_subject.colombia_economics.filtered_monthly
),

labor_summary AS (
    SELECT
        Period,
        ROUND(AVG(OccupationRate), 2) AS AvgOccupationRate,
        ROUND(AVG(UnemploymentRate), 2) AS AvgUnemploymentRate
    FROM labor
    GROUP BY Period
),

gdp_summary AS (
    SELECT
        CASE
            WHEN SUBSTRING(Quarter, 1, 4) <= '2019'
                THEN 'Pre-COVID'
            WHEN SUBSTRING(Quarter, 1, 4) = '2020'
                THEN 'COVID'
            ELSE 'Post-COVID'
        END AS Period,
        ROUND(AVG(RealGDPGrowth), 2) AS AvgGDPGrowth
    FROM bi_subject.colombia_economics.filtered_quarterly
    GROUP BY
        CASE
            WHEN SUBSTRING(Quarter, 1, 4) <= '2019'
                THEN 'Pre-COVID'
            WHEN SUBSTRING(Quarter, 1, 4) = '2020'
                THEN 'COVID'
            ELSE 'Post-COVID'
        END
)

SELECT
    l.Period,
    l.AvgOccupationRate,
    l.AvgUnemploymentRate,
    g.AvgGDPGrowth
FROM labor_summary l
JOIN gdp_summary g
    ON l.Period = g.Period
ORDER BY
    CASE l.Period
        WHEN 'Pre-COVID' THEN 1
        WHEN 'COVID' THEN 2
        WHEN 'Post-COVID' THEN 3
    END;

Period,AvgOccupationRate,AvgUnemploymentRate,AvgGDPGrowth
Pre-COVID,58.42,10.43,2.87
COVID,50.43,16.67,-7.17
Post-COVID,56.87,10.64,4.54


## Stock market: Which periods produced the strongest and weakest MSCI COLCAP monthly returns?

In [0]:
%sql
WITH ranked_returns AS (
    SELECT
        MonthlyDate,
        ROUND(MonthlyReturn, 2) AS MonthlyReturn,
        ROW_NUMBER() OVER (ORDER BY MonthlyReturn DESC) AS BestRank,
        ROW_NUMBER() OVER (ORDER BY MonthlyReturn ASC) AS WorstRank
    FROM bi_subject.colombia_economics.filtered_monthly
    WHERE MonthlyReturn IS NOT NULL
)

SELECT
    CASE
        WHEN BestRank <= 5 THEN 'Strongest'
        WHEN WorstRank <= 5 THEN 'Weakest'
    END AS ReturnType,
    MonthlyDate,
    MonthlyReturn
FROM ranked_returns
WHERE BestRank <= 5
   OR WorstRank <= 5
ORDER BY
    CASE WHEN ReturnType = 'Strongest' THEN 1 ELSE 2 END,
    MonthlyReturn DESC;

ReturnType,MonthlyDate,MonthlyReturn
Strongest,2026-01,19.67
Strongest,2020-12,14.3
Strongest,2020-11,10.67
Strongest,2025-01,10.3
Strongest,2022-10,9.15
Weakest,2023-05,-8.32
Weakest,2023-08,-8.39
Weakest,2026-02,-10.18
Weakest,2022-06,-17.49
Weakest,2020-03,-27.48


## Relationships: Is there an observable relationship between inflation, the policy rate, TRM, and MSCI COLCAP returns?

In [0]:
%sql
SELECT
    ROUND(CORR(InflationYOY, PolicyRate), 3) AS Inflation_PolicyRate,
    ROUND(CORR(InflationYOY, TRMAvg), 3) AS Inflation_TRM,
    ROUND(CORR(InflationYOY, MonthlyReturn), 3) AS Inflation_COLCAPReturn,
    ROUND(CORR(PolicyRate, TRMAvg), 3) AS PolicyRate_TRM,
    ROUND(CORR(PolicyRate, MonthlyReturn), 3) AS PolicyRate_COLCAPReturn,
    ROUND(CORR(TRMAvg, MonthlyReturn), 3) AS TRM_COLCAPReturn
FROM bi_subject.colombia_economics.filtered_monthly
WHERE InflationYOY IS NOT NULL
  AND PolicyRate IS NOT NULL
  AND TRMAvg IS NOT NULL
  AND MonthlyReturn IS NOT NULL;

Inflation_PolicyRate,Inflation_TRM,Inflation_COLCAPReturn,PolicyRate_TRM,PolicyRate_COLCAPReturn,TRM_COLCAPReturn
0.756,0.719,-0.066,0.566,0.055,-0.024
